# Calibrating gate frequencies with FaultTune

This notebook is the Python version of the worked example of FaultTune, a Rust implementation
of the **Snake Optimizer** (Klimov et al., [arXiv:2006.04594](https://arxiv.org/abs/2006.04594))
with a Python interface. It is the same program as `examples/gate_frequency.py`.

The Snake learns the control parameters of a processor whose parts interact: it walks through
the graph of the device one small neighbourhood at a time, optimizes the parameters of the
neighbourhood with the already calibrated neighbours as constraints, freezes them and moves on.

Here the parameters are the **gate frequencies** of an 8x8 lattice of qubits: one frequency
per qubit (single-qubit gate) and one per coupler (two-qubit gate). We will

1. calibrate the whole device and watch the progress,
2. compare settings (`parameter_distance`, traversal heuristic) with a naive baseline,
3. re-calibrate locally after one qubit drifted,
4. save and load the result as JSON,
5. checkpoint a run in a journal and resume it after an interruption,
6. stitch four independently calibrated quadrants together.

The calibration is deterministic: the same inputs and seed give the same outputs, whatever the
number of threads.

Run the next cell to install the package into the notebook's environment, or skip it if
`faulttune` is already installed (for example with `maturin develop`).

In [ ]:
%pip install --quiet faulttune numpy

In [ ]:
import tempfile
import threading
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
from numpy.typing import NDArray

import faulttune as ft

# The size of the lattice: 64 qubits, 112 couplers, 98 parasitic diagonals.
ROWS = 8
COLS = 8

## The error model

FaultTune provides the Snake machinery (graph, neighbourhoods, traversal, parallelism, the
parameter database). **You provide the physics**: a subclass of `faulttune.ErrorModel` with

* `options(element)`: the candidate values of one element (here 100 frequencies between 5 and
  7 GHz),
* `error(step, candidates)`: the estimated error of many candidate assignments **at once**.
  `candidates` is a `float64` array of shape `(n_candidates, len(step.parameters))`, and
  `step.constraints` maps the already calibrated neighbours to their fixed values. The method
  returns an array of shape `(n_candidates,)`, with `inf` for forbidden candidates,
* optionally `prepare(step)`: collect the data a step needs (for example by measuring
  hardware). Whatever it returns is available as `step.data` in `error`.

Receiving a whole batch keeps the cost of calling into Python low. The toy model below has

* **relaxation** defects: every qubit has three random Lorentzian bumps of error, and a
  two-qubit gate pays for both of its qubits,
* a **detuning** cost: a two-qubit gate prefers a frequency near the mean of its two qubits,
* **crosstalk** between elements within distance 2 of each other (a Lorentzian in the frequency
  difference, with a random coupling strength).

Our `prepare` resolves the neighbours, couplings and endpoint frequencies of a step once, into a
`StepPlan`, so `error` only does array arithmetic.

In [ ]:
@dataclass(frozen=True)
class StepPlan:
    """Everything one step needs to evaluate many candidates, resolved once in `prepare`.

    Looking up neighbours, couplings and endpoint frequencies is cheap but not free; doing it
    once per step instead of once per `error` call is the typical use of `prepare` and
    `step.data`.
    """

    #: Per parameter: defect centres, squared half widths and strengths (arrays of one size).
    defects: tuple[tuple[NDArray[np.float64], NDArray[np.float64], NDArray[np.float64]], ...]
    #: Per two-qubit gate (a column): the sources of its endpoint qubits' frequencies, each
    #: either ``("fixed", value)`` (a constraint) or ``("column", j)`` (another parameter).
    detuning: tuple[tuple[int, tuple[tuple[str, float], ...]], ...]
    #: ``pairs[j, l]`` is the crosstalk coupling of parameters j < l that interact, else 0.
    pairs: NDArray[np.float64]
    #: ``with_constraints[j, c]`` is the coupling of parameter j and constraint c, else 0.
    with_constraints: NDArray[np.float64]
    #: The fixed frequencies of the constraints, in the order of the columns above.
    constraint_values: NDArray[np.float64]


class GateFrequencyModel(ft.ErrorModel):
    """A toy model of gate frequencies on a qubit lattice.

    Nodes are single-qubit gate frequencies, engineered edges are two-qubit gate frequencies.
    Each element chooses among 100 frequencies in [5, 7] GHz. The error of an element is a
    smooth sum of

    * **relaxation**: every qubit has three random defects, each a Lorentzian
      ``s w^2 / ((f - c)^2 + w^2)`` with centre ``c`` in [5, 7] GHz, half width ``w`` of
      5 to 20 MHz and strength ``s`` in [0.5, 2]. A single-qubit gate pays for the defects of its
      qubit, a two-qubit gate for those of both of its qubits, at the gate frequency;
    * **detuning**: a two-qubit gate prefers a frequency near the mean of its two qubits'
      frequencies (``(f - f_ref)^2``, weight 1 per GHz^2), where known;
    * **crosstalk**: two elements within distance 2 of each other (in the sense of the paper) pay
      ``g w_x^2 / ((f_a - f_b)^2 + w_x^2)`` with ``w_x`` = 40 MHz and a random coupling ``g`` in
      [0.2, 1].

    The error of a candidate is the own terms of the step's parameters, plus the crosstalk of
    every interacting pair among the parameters and of every interacting (parameter, constraint)
    pair. It is a pure function of the step that only reads the model's data, so it is safe with
    any cache policy, and calls from the several subgoals of a run may overlap.
    """

    NUM_OPTIONS = 100
    FREQUENCY_MIN = 5.0
    FREQUENCY_MAX = 7.0
    DEFECTS_PER_QUBIT = 3
    INTERACTION_DISTANCE = 2
    CROSSTALK_WIDTH = 0.04
    DETUNING_WEIGHT = 1.0
    BASELINE_FREQUENCY = 6.0

    def __init__(self, graph: ft.ProcessorGraph, seed: int = 53) -> None:
        self.graph = graph
        elements = graph.all_elements()
        n = len(elements)
        rng = np.random.default_rng(seed)

        # Random defects of every qubit, kept per element: a node owns its own, an edge owns
        # those of both endpoints.
        qubit_defects: dict[int, tuple[Any, Any, Any]] = {}
        for node in graph.nodes():
            centre = rng.uniform(self.FREQUENCY_MIN, self.FREQUENCY_MAX, self.DEFECTS_PER_QUBIT)
            width = rng.uniform(0.005, 0.020, self.DEFECTS_PER_QUBIT)
            strength = rng.uniform(0.5, 2.0, self.DEFECTS_PER_QUBIT)
            qubit_defects[node.index] = (centre, width * width, strength)
        self.defects: list[
            tuple[NDArray[np.float64], NDArray[np.float64], NDArray[np.float64]]
        ] = []
        for element in elements:
            owners = [element] if element.is_node else list(element.endpoints or ())
            parts = [qubit_defects[o.index] for o in owners]
            self.defects.append(
                (
                    np.concatenate([p[0] for p in parts]),
                    np.concatenate([p[1] for p in parts]),
                    np.concatenate([p[2] for p in parts]),
                )
            )

        # Symmetric random couplings of all pairs, and which pairs interact at all.
        coupling = np.triu(rng.uniform(0.2, 1.0, (n, n)), k=1)
        self.coupling = coupling + coupling.T
        self.interacts = self._interaction_matrix(graph, self.INTERACTION_DISTANCE)

        self.options_ = np.linspace(self.FREQUENCY_MIN, self.FREQUENCY_MAX, self.NUM_OPTIONS)

    @staticmethod
    def _interaction_matrix(graph: ft.ProcessorGraph, distance: int) -> NDArray[np.bool_]:
        """Which pairs of elements are within `distance` of each other (paper sense).

        Distances are counted on the incidence graph (each node linked to its incident edges):
        an element is within `d` of another if the other is within incidence distance
        ``2d`` (for an edge) or ``2d - 1`` (for a node) of it. The relation is symmetric.
        """
        elements = graph.all_elements()
        neighbours: list[list[int]] = [[] for _ in elements]
        for edge in graph.edges():
            for node in edge.endpoints or ():
                neighbours[edge.index].append(node.index)
                neighbours[node.index].append(edge.index)
        interacts = np.zeros((len(elements), len(elements)), dtype=bool)
        for element in elements:
            radius = 2 * distance if element.is_edge else max(2 * distance - 1, 0)
            seen = {element.index}
            frontier = [element.index]
            for _ in range(radius):
                frontier = [j for i in frontier for j in neighbours[i] if j not in seen]
                seen.update(frontier)
            interacts[element.index, sorted(seen)] = True
            interacts[element.index, element.index] = False
        assert (interacts == interacts.T).all(), "interaction must be symmetric"
        return interacts

    # -- the ErrorModel interface -------------------------------------------------------------

    def options(self, element: ft.Element) -> NDArray[np.float64]:
        """The 100 candidate frequencies of a node or an engineered edge."""
        if element.kind not in (ft.ElementKind.NODE, ft.ElementKind.ENGINEERED_EDGE):
            raise ValueError(f"{element.label} has no gate frequency; exclude it from the goal")
        return self.options_

    def prepare(self, step: ft.Step) -> StepPlan:
        """Resolves the step once; the result is `step.data` in every `error` call."""
        return self.plan(step.parameters, step.constraints)

    def error(self, step: ft.Step, candidates: NDArray[np.float64]) -> NDArray[np.float64]:
        """The errors of all candidates at once: shape (n_candidates,)."""
        return self.evaluate(step.data, candidates)

    # -- the physics --------------------------------------------------------------------------

    def plan(self, parameters: list[ft.Element], constraints: dict[ft.Element, float]) -> StepPlan:
        """Resolves which defects, couplings and endpoint frequencies a step involves."""
        ids = [p.index for p in parameters]
        column = {i: j for j, i in enumerate(ids)}
        fixed = {c.index: float(f) for c, f in constraints.items()}

        detuning = []
        for j, parameter in enumerate(parameters):
            if not parameter.is_edge:
                continue
            sources: list[tuple[str, float]] = []
            for end in parameter.endpoints or ():
                if end.index in fixed:
                    sources.append(("fixed", fixed[end.index]))
                elif end.index in column:
                    sources.append(("column", float(column[end.index])))
            detuning.append((j, tuple(sources)))

        constraint_ids = list(fixed)
        pairs = np.triu(self.coupling[np.ix_(ids, ids)] * self.interacts[np.ix_(ids, ids)], k=1)
        with_constraints = (
            self.coupling[np.ix_(ids, constraint_ids)] * self.interacts[np.ix_(ids, constraint_ids)]
        )
        return StepPlan(
            defects=tuple(self.defects[i] for i in ids),
            detuning=tuple(detuning),
            pairs=pairs,
            with_constraints=with_constraints,
            constraint_values=np.array([fixed[c] for c in constraint_ids], dtype=np.float64),
        )

    def evaluate(self, plan: StepPlan, candidates: NDArray[np.float64]) -> NDArray[np.float64]:
        """The error of every row of `candidates` (float64, shape (n_candidates, n_parameters))."""
        total = np.zeros(candidates.shape[0])
        w2 = self.CROSSTALK_WIDTH**2
        for j, (centre, width_sq, strength) in enumerate(plan.defects):
            d = candidates[:, j, None] - centre
            total += (strength * width_sq / (d * d + width_sq)).sum(axis=1)
        for j, sources in plan.detuning:
            known = [f if kind == "fixed" else candidates[:, int(f)] for kind, f in sources]
            if known:
                reference = 0.5 * (known[0] + known[1]) if len(known) == 2 else known[0]
                total += self.DETUNING_WEIGHT * (candidates[:, j] - reference) ** 2
        if plan.pairs.any():
            d = candidates[:, :, None] - candidates[:, None, :]
            total += (plan.pairs * w2 / (d * d + w2)).sum(axis=(1, 2))
        if plan.with_constraints.size:
            d = candidates[:, :, None] - plan.constraint_values
            total += (plan.with_constraints * w2 / (d * d + w2)).sum(axis=(1, 2))
        return total

    # -- measuring a result -------------------------------------------------------------------

    def assignment_error(self, values: dict[ft.Element, float]) -> float:
        """The full-model error of an assignment: every own term and every interacting pair."""
        parameters = list(values)
        plan = self.plan(parameters, {})
        row = np.array([[values[p] for p in parameters]])
        return float(self.evaluate(plan, row)[0])

    def mean_error(self, db: ft.ParameterDatabase) -> float:
        """Mean error per calibrated gate."""
        values = db.to_dict()
        return self.assignment_error(values) / len(values) if values else 0.0

    def baseline_mean_error(self, db: ft.ParameterDatabase) -> float:
        """The mean error if every gate of `db` ran at 6 GHz, the middle of the band."""
        values = dict.fromkeys(db.to_dict(), self.BASELINE_FREQUENCY)
        return self.assignment_error(values) / len(values) if values else 0.0

## The device

A `ProcessorGraph` holds the nodes (qubits) and edges. `square_lattice` builds the usual layout:
nearest-neighbour **engineered** edges (the couplers used for two-qubit gates) and diagonal
**parasitic** edges that have no parameters but carry crosstalk. The **goal** is the set of
elements to calibrate: everything except the parasitic edges.

The **algorithm subgraphs** say which elements can be active at the same time in the algorithm
that will run on the processor (the five layers of the cross-entropy benchmark, XEB, here);
only elements that can be active together constrain each other. Without them, everything is
assumed to run at once.

In [ ]:
graph = ft.ProcessorGraph.square_lattice(ROWS, COLS)
goal = graph.goal(exclude=[ft.ElementKind.PARASITIC_EDGE])
xeb = ft.AlgorithmSubgraphs.xeb_square_lattice(graph, ROWS, COLS)
model = GateFrequencyModel(graph, seed=53)

print(
    f"Device: {ROWS}x{COLS} lattice, {len(graph.all_elements())} elements, "
    f"{len(goal)} of them to calibrate, {len(xeb.subgraphs)} algorithm subgraphs"
)


def make_snake(parameter_distance: int = 1, **settings: Any) -> ft.Snake:
    """A Snake for this device: the XEB setting with a chosen d_P."""
    settings.setdefault("constraint_distance", 2)  # d_R
    settings.setdefault("traversal_distance", 2)  # d_T
    return ft.Snake(
        graph, goal, parameter_distance=parameter_distance, algorithm=xeb, **settings
    )

## 1. Calibrate, with progress

The Snake has three distances. `parameter_distance` (d_P) is how many parameters are optimized
together in one step, `constraint_distance` (d_R) how far calibrated elements can be and still
constrain the step, and `traversal_distance` (d_T) how far the Snake can move between steps.
The values below are those of the paper's XEB example.

`on_step` is called on the calling thread for every committed step, which is the place for a
progress bar. `record.index` is the ordinal of the step within its subgoal.

In [ ]:
def progress(record: ft.StepRecord) -> None:
    if record.index % 8 == 0:
        print(
            f"    step {record.index:>3}: centre {record.centre.label:<10} "
            f"{len(record.parameters)} parameters, {len(record.constraints):>2} constraints, "
            f"error {record.error:.3f}"
        )


snake = make_snake()
calibrated = snake.calibrate(model, on_step=progress)
report = snake.last_report
assert report is not None
full_steps = report.steps
print(
    f"Done: {report.steps} steps in {report.passes} pass(es), {report.evaluations} "
    f"evaluations; calibrated {len(calibrated)} of {len(goal)} elements."
)
print("Worst steps (largest error), the natural candidates for re-optimization:")
for step in report.worst_steps[:3]:
    print(f"  centre {step.centre.label:<10} error {step.error:.3f}")

The result is a `ParameterDatabase`: a mapping from elements to values, plus the ordered history
of the steps that set them. Here are the qubit frequencies (GHz) laid out like the lattice:

In [ ]:
for row in range(ROWS):
    print(" ".join(f"{calibrated[graph.node(row, col)]:5.2f}" for col in range(COLS)))

## 2. Compare with a baseline and with other settings

`model.mean_error` evaluates the full model on a database: the own terms of every calibrated
element plus the crosstalk of every interacting pair, divided by the number of gates. The naive
baseline runs every gate at 6 GHz, in the middle of the band, where every interacting pair
sits at its full crosstalk peak.

`parameter_distance` is the key trade-off between speed and quality: `d_P = 0` optimizes one
parameter at a time over its 100 options (fast, but the reachable solutions are few); larger
values optimize a whole neighbourhood together. Traversal heuristics are given by name.

In [ ]:
baseline = model.baseline_mean_error(calibrated)
print(f"  {'setting':<34} {'steps':>7} {'evaluations':>12} {'mean error':>11}")
print(f"  {'naive: every gate at 6 GHz':<34} {'-':>7} {'-':>12} {baseline:>11.4f}")

settings = [
    ("d_P = 0, breadth-first", 0, "breadth_first"),
    ("d_P = 1, breadth-first (default)", 1, "breadth_first"),
    ("d_P = 2, breadth-first", 2, "breadth_first"),
    ("d_P = 1, depth-first", 1, "depth_first"),
    ("d_P = 1, random", 1, "random"),
]
means = {}
for name, parameter_distance, heuristic in settings:
    other = make_snake(parameter_distance, heuristic=heuristic)
    db = other.calibrate(model)
    other_report = other.last_report
    assert other_report is not None
    means[name] = model.mean_error(db)
    print(
        f"  {name:<34} {other_report.steps:>7} {other_report.evaluations:>12} "
        f"{means[name]:>11.4f}"
    )

A larger `d_P` means fewer, bigger steps and a lower error, at the price of more evaluations.
Every setting is far below the naive baseline.

In [ ]:
assert all(mean < baseline / 5 for mean in means.values())
assert means["d_P = 2, breadth-first"] < means["d_P = 0, breadth-first"]
print(
    f"d_P = 2 has {100 * (1 - means['d_P = 2, breadth-first'] / means['d_P = 0, breadth-first']):.0f}% "
    f"lower error than d_P = 0; the worst setting beats the baseline by a factor of "
    f"{baseline / max(means.values()):.0f}."
)

## 3. Local re-calibration after drift

Calibrated values drift. When an element's performance falls past a threshold, its calibration
has expired: `snake.recalibrate(database, element, model)` forgets the neighbourhood of the
element (`expiry_radius`, 1 by default: the element and its incident engineered edges) and runs
the same algorithm again, so only that region is re-learned, subject to all the surrounding
gates, which stay untouched.

We simulate drift by wrapping the model: qubit `q3_4` gets a new defect (a two-level system)
right at its operating frequency. `DriftedModel` delegates to the base model and adds the new
Lorentzian.

In [ ]:
class DriftedModel(ft.ErrorModel):
    """The model plus one new defect (a two-level system) near a chosen qubit.

    The relaxation error of that qubit, and of the gates it takes part in, gets one more
    Lorentzian bump. It shows how a model is wrapped to describe a device that changed: the
    wrapper delegates to the base model and adds the new term.
    """

    WIDTH = 0.015  #: half width of the new defect (GHz)
    STRENGTH = 3.0  #: height of the bump

    def __init__(self, base: GateFrequencyModel, qubit: ft.Element, centre: float) -> None:
        self.base = base
        self.qubit = qubit
        self.centre = centre

    def options(self, element: ft.Element) -> NDArray[np.float64]:
        return self.base.options(element)

    def prepare(self, step: ft.Step) -> StepPlan:
        return self.base.prepare(step)

    def penalty(self, element: ft.Element, frequencies: NDArray[np.float64]) -> NDArray[np.float64]:
        """The extra error of `element` at `frequencies` (zero unless it touches the qubit)."""
        touches = element == self.qubit or self.qubit in (element.endpoints or ())
        if not touches:
            return np.zeros_like(frequencies)
        d = frequencies - self.centre
        return self.STRENGTH * self.WIDTH**2 / (d * d + self.WIDTH**2)

    def error(self, step: ft.Step, candidates: NDArray[np.float64]) -> NDArray[np.float64]:
        total = self.base.error(step, candidates)
        for j, parameter in enumerate(step.parameters):
            total = total + self.penalty(parameter, candidates[:, j])
        return total

    def total_penalty(self, db: ft.ParameterDatabase) -> float:
        """The summed extra error of every calibrated element of `db`."""
        return float(sum(self.penalty(e, np.array([v]))[0] for e, v in db.to_dict().items()))

In [ ]:
db = calibrated.copy()  # keep `calibrated` for later
qubit = graph.node(3, 4)
stale_frequency = db[qubit]
drifted = DriftedModel(model, qubit, centre=stale_frequency)
print(f"{qubit.label} runs at {stale_frequency:.3f} GHz, and a new defect appears exactly there.")

before = db.copy()
stale_penalty = drifted.total_penalty(db)
history_before = len(db.history)

report = make_snake().recalibrate(db, qubit, drifted)

forgotten = {qubit} | {e for e in goal if e.is_edge and qubit in (e.endpoints or ())}
old, new = before.to_dict(), db.to_dict()
changed = [e for e in graph.all_elements() if old.get(e) != new.get(e)]
assert set(changed) <= forgotten, "an entry outside the neighbourhood changed"
assert goal <= db.calibrated, "the goal is no longer fully calibrated"

print(f"Forgot {len(forgotten)} elements and re-learned them in {report.steps} step(s), "
      f"{report.evaluations} evaluations.")
print("Entries that changed value:")
for e in changed:
    print(f"  {e.label:<12} {old[e]:.3f} GHz -> {new[e]:.3f} GHz")
print(
    f"{len(db) - len(changed)} of the {len(db)} calibrated entries are untouched; the history "
    f"grew from {history_before} to {len(db.history)} records "
    "(one Invalidation, then the new steps)."
)
fresh_penalty = drifted.total_penalty(db)
print(
    f"Extra error caused by the defect, summed over all gates: {stale_penalty:.3f} with the "
    f"stale calibration, {fresh_penalty:.3f} after re-calibration."
)
assert qubit in changed and fresh_penalty < stale_penalty

## 4. Save and load

A database saves to JSON, keyed by element label, so the file is readable and diffs well. It
records a fingerprint of the graph: loading it onto a different device fails with
`GraphMismatchError` instead of silently attaching values to the wrong elements.

In [ ]:
scratch_dir = tempfile.TemporaryDirectory(prefix="faulttune-example-")
scratch = Path(scratch_dir.name)

path = scratch / "calibration.json"
calibrated.save(path)
print(f"Saved {len(calibrated)} entries and {len(calibrated.history)} history records. "
      "The start of the file:")
for line in path.read_text().splitlines()[:12]:
    print(f"  {line}")
print("  ...")

loaded = ft.ParameterDatabase.load(path, graph)
print("Loaded it back: same values and history (bit for bit):", loaded.same_outcome(calibrated))
assert loaded.same_outcome(calibrated)

other_device = ft.ProcessorGraph.square_lattice(4, 4)
try:
    ft.ParameterDatabase.load(path, other_device)
except ft.GraphMismatchError as err:
    print(f"Loading it onto a 4x4 lattice is refused: {err}")
else:
    raise AssertionError("a calibration was loaded onto the wrong graph")

## 5. Checkpoint and resume

When `prepare` measures hardware, a full run takes hours, and a crash should not lose it.
`checkpoint=path` appends every committed step to a **journal**, one JSON line each, flushed as
it is written. After an interruption (Ctrl-C raises `KeyboardInterrupt`, `snake.cancel()` raises
`CancelledError`; both keep every committed step in the database), a fresh process rebuilds the
state with `ParameterDatabase.replay_journal` and continues with `calibrate(database=...)`.

To simulate the interruption, `on_step` cancels the run after 25 steps. A real step takes
seconds, so `on_step` is always ahead of the engine; the toy model is faster than the callback,
so `HoldingModel` makes the step wait until the callback has cancelled. This only makes the
notebook deterministic.

In [ ]:
class HoldingModel(ft.ErrorModel):
    """The model, but at one step its `error` announces itself and waits to be released.

    A real calibration measures hardware: each step takes seconds, so `on_step` (which runs on
    the main thread as soon as a step is committed) is always ahead of the engine. The toy model
    is so fast that the engine could finish the whole run before the callback gets to cancel
    it, and a cancelled run would stop after a different number of steps every time. This
    wrapper makes the example deterministic: at step `hold_step` it sets `reached` and then
    waits for `release`, so the callback can cancel while that step is known to be running.
    """

    def __init__(self, base: GateFrequencyModel, hold_step: int) -> None:
        self.base = base
        self.hold_step = hold_step
        self.reached = threading.Event()
        self.release = threading.Event()

    def options(self, element: ft.Element) -> NDArray[np.float64]:
        return self.base.options(element)

    def prepare(self, step: ft.Step) -> StepPlan:
        return self.base.prepare(step)

    def error(self, step: ft.Step, candidates: NDArray[np.float64]) -> NDArray[np.float64]:
        if step.index == self.hold_step:
            self.reached.set()
            if not self.release.wait(60):
                raise TimeoutError("on_step never released the step")
        return self.base.error(step, candidates)

In [ ]:
journal = scratch / "run.journal"
stop_after = 25  # the run is interrupted during step 25: 26 steps are committed
snake = make_snake()
holding = HoldingModel(model, hold_step=stop_after)


def stop_early(record: ft.StepRecord) -> None:
    if record.index + 1 == stop_after:
        holding.reached.wait(60)
        snake.cancel()  # Ctrl-C does the same
        holding.release.set()


interrupted = ft.ParameterDatabase(graph)
try:
    snake.calibrate(holding, database=interrupted, on_step=stop_early, checkpoint=journal)
except ft.CancelledError as err:
    assert err.partial is interrupted
    print(f"Run cancelled after {interrupted.step_count} steps; the database kept all of them "
          f"({len(interrupted)} of {len(goal)} elements).")
else:
    raise AssertionError("the run finished before it was cancelled")
assert interrupted.step_count == stop_after + 1

A crash in the middle of writing a line leaves a torn last line, which is ignored on replay (a
crash loses at most the step being written). We append one by hand, then pretend to be a new
process: nothing is in memory but the journal. `read_journal` returns the step records (with
option indices, not values); `replay_journal` applies them to a database, asking the model only
for its options.

In [ ]:
with journal.open("a") as f:
    f.write('{"type":"step","pass":0,"subgoal":0,"centre":"q4')

steps = ft.read_journal(journal, graph)
restored = ft.ParameterDatabase(graph)
applied = restored.replay_journal(journal, model)
identical = restored.same_outcome(interrupted)
print(f"Journal read (the torn last line is ignored): {len(steps)} steps, {applied} replayed; "
      f"the replayed database equals the interrupted one: {identical}")
assert identical and len(steps) == applied == interrupted.step_count

Resuming is the same call as a fresh calibration, on the partly filled database. The journal is
appended to (and the torn line repaired), so it stays complete. A resumed run starts a fresh
traversal from the remaining elements, so it need not equal an uninterrupted run, only complete
it.

In [ ]:
resumer = make_snake()
resumed = resumer.calibrate(model, database=restored, checkpoint=journal)
resumed_report = resumer.last_report
assert resumed is restored and goal <= restored.calibrated and resumed_report is not None
print(f"Resumed: {resumed_report.steps} more steps ({full_steps} "
      f"in an uninterrupted run), all {len(goal)} goal elements calibrated; "
      f"mean error per gate {model.mean_error(restored):.4f}.")

# The journal alone rebuilds the final state: a third party can do it.
rebuilt = ft.ParameterDatabase(graph)
rebuilt.replay_journal(journal, model)
assert rebuilt.same_outcome(restored)

## 6. Stitching

A large processor can be split on purpose into regions that cannot constrain each other,
calibrated in parallel, and then **stitched** together by calibrating the gap between them.
`stitch_plan` computes the plan: stage one calibrates the cores of the regions (the parts more
than `max(d_P, d_R)` apart), stage two the gap, under the constraints from both sides.

Here the regions are the four quadrants of the lattice, picked with the layout coordinates of
the graph (edges sit at the midpoint of their endpoints).

In [ ]:
coords = graph.coords
assert coords is not None
mid_x, mid_y = (COLS - 1) / 2, (ROWS - 1) / 2
groups: list[list[ft.Element]] = [[], [], [], []]
for element in goal:
    x, y = coords[element.index]
    groups[int(x > mid_x) + 2 * int(y > mid_y)].append(element)
regions = [graph.element_set(g) for g in groups]

snake = make_snake()
plan = snake.stitch_plan(regions)
print(f"Quadrants hold {[len(r) for r in regions]} elements. Stage one calibrates "
      f"{len(plan.stage_one)} of them, stage two the {len(plan.gap)} elements of the gap.")

stitched = ft.ParameterDatabase(graph)
stitch_report = snake.stitch(stitched, plan, model)
subgoals = sorted({(s.pass_, s.subgoal) for s in stitched.steps()})
print(f"Stitched calibration: {stitch_report.steps} steps in {stitch_report.passes} passes and "
      f"{len(subgoals)} subgoals, mean error per gate {model.mean_error(stitched):.4f} "
      f"(unstitched: {model.mean_error(calibrated):.4f}).")
assert goal <= stitched.calibrated

The result does not depend on the number of threads, history included. (Python error models
take the GIL, so the subgoals of a small NumPy model hardly overlap; with a model that waits for
hardware or does large array operations they run side by side.)

In [ ]:
databases = []
for threads in (1, 4):
    db = ft.ParameterDatabase(graph)
    make_snake(threads=threads).stitch(db, plan, model)
    databases.append(db)
identical = databases[0].same_outcome(databases[1]) and databases[0].same_outcome(stitched)
print("Identical databases with 1 and 4 threads, history included:", identical)
assert identical
scratch_dir.cleanup()

## Where to go from here

* Replace `GateFrequencyModel` with your own: `options` and `error` are all you need. Do the work
  in NumPy (it releases the GIL in large array operations) or in Rust for heavy models.
* Custom traversal heuristics, seed policies and step optimizers can be given as Python
  callables or by name (`"coordinate_descent"`, `"exhaustive"`, `"random_search"`).
* `on_model_error="skip"` or `("retry", n)` keeps a run going when a measurement fails.
* See the README and `docs/ARCHITECTURE.md` for the design.